In [1]:
db = "./movies.db"

In [2]:
%load_ext sql
%config SqlMagic.feedback=False
%sql sqlite:///{db}?check_same_thread=false
%sql SELECT name FROM sqlite_master WHERE type='table';

 * sqlite:///./movies.db?check_same_thread=false


name
movies
people
ratings
stars


In [4]:
%sql SELECT * FROM "movies" LIMIT 10;

 * sqlite:///./movies.db?check_same_thread=false


id,title,year
11801,Tötet nicht mehr,2019
13274,Istoriya grazhdanskoy voyny,2021
15414,La tierra de los toros,2000
15724,Dama de noche,1993
31458,El huésped del sevillano,1970
35423,Kate & Leopold,2001
36606,"Another Time, Another Place",1983
38086,Shiva und die Galgenblume,1993
38687,Let There Be Light,1980
39442,"Habla, mudita",1973


In [5]:
%%sql
SELECT * FROM "movies"
WHERE "title" = 'Cars';

 * sqlite:///./movies.db?check_same_thread=false


id,title,year
317219,Cars,2006


In [10]:
%%sql
SELECT * FROM sqlite_master;

 * sqlite:///./movies.db?check_same_thread=false


type,name,tbl_name,rootpage,sql
table,movies,movies,2,"CREATE TABLE ""movies"" ( ""id"" INTEGER, ""title"" TEXT NOT NULL, ""year"" NUMERIC, PRIMARY KEY(""id""))"
table,people,people,3,"CREATE TABLE ""people"" ( ""id"" INTEGER, ""name"" TEXT NOT NULL, ""birth"" NUMERIC, PRIMARY KEY(""id""))"
table,ratings,ratings,4,"CREATE TABLE ""ratings"" ( ""id"" INTEGER, ""movie_id"" INTEGER UNIQUE, ""rating"" REAL NOT NULL, ""votes"" INTEGER NOT NULL, PRIMARY KEY(""id""), FOREIGN KEY(""movie_id"") REFERENCES ""movies""(""id""))"
index,sqlite_autoindex_ratings_1,ratings,5,None
table,stars,stars,6,"CREATE TABLE ""stars"" ( ""movie_id"" INTEGER, ""person_id"" INTEGER, PRIMARY KEY(""movie_id"", ""person_id""), FOREIGN KEY(""movie_id"") REFERENCES ""movies""(""id""), FOREIGN KEY(""person_id"") REFERENCES ""people""(""id""))"
index,sqlite_autoindex_stars_1,stars,7,None


`$ .timer on` for timing a query.

In [ ]:
%%sql
SELECT * FROM "movies"
WHERE "title" = 'Cars';

 * sqlite:///./movies.db?check_same_thread=false


id,title,year
317219,Cars,2006


In [14]:
%%sql
CREATE INDEX "title_index"
ON "movies" ("title");

 * sqlite:///./movies.db?check_same_thread=false


[]

In [15]:
%%sql
EXPLAIN QUERY PLAN
SELECT * FROM MOVIES WHERE "title" = 'Cars';

 * sqlite:///./movies.db?check_same_thread=false


id,parent,notused,detail
3,0,63,SEARCH MOVIES USING INDEX title_index (title=?)


In [16]:
%sql DROP INDEX "title_index";

 * sqlite:///./movies.db?check_same_thread=false


[]

In [3]:
%%sql
EXPLAIN QUERY PLAN
SELECT * FROM MOVIES WHERE "title" = 'Cars';

 * sqlite:///./movies.db?check_same_thread=false


id,parent,notused,detail
2,0,216,SCAN MOVIES


In [4]:
%%sql
SELECT "title" FROM "movies"
WHERE "id" IN (
    SELECT "movie_id" FROM "stars"
    WHERE "person_id" = (
        SELECT "id" FROM "people"
        WHERE "name" = 'Tom Hanks'
    )
);

 * sqlite:///./movies.db?check_same_thread=false


title
Bachelor Party
Splash
The Man with One Red Shoe
Volunteers
Every Time We Say Goodbye
The Money Pit
Nothing in Common
Dragnet
Big
Punchline


In [5]:
%%sql
EXPLAIN QUERY PLAN
SELECT "title" FROM "movies"
WHERE "id" IN (
    SELECT "movie_id" FROM "stars"
    WHERE "person_id" = (
        SELECT "id" FROM "people"
        WHERE "name" = 'Tom Hanks'
    )
);

 * sqlite:///./movies.db?check_same_thread=false


id,parent,notused,detail
2,0,91,SEARCH movies USING INTEGER PRIMARY KEY (rowid=?)
6,0,0,LIST SUBQUERY 2
9,6,216,SCAN stars
14,6,0,SCALAR SUBQUERY 1
18,14,216,SCAN people
31,6,0,CREATE BLOOM FILTER


In [6]:
%%sql
CREATE INDEX "person_index"
ON "stars" ("person_id");

 * sqlite:///./movies.db?check_same_thread=false


[]

In [8]:
%%sql
CREATE INDEX "name_index"
ON "people" ("name");

 * sqlite:///./movies.db?check_same_thread=false


[]

In [9]:
%%sql
EXPLAIN QUERY PLAN
SELECT "title" FROM "movies"
WHERE "id" IN (
    SELECT "movie_id" FROM "stars"
    WHERE "person_id" = (
        SELECT "id" FROM "people"
        WHERE "name" = 'Tom Hanks'
    )
);

 * sqlite:///./movies.db?check_same_thread=false


id,parent,notused,detail
2,0,91,SEARCH movies USING INTEGER PRIMARY KEY (rowid=?)
6,0,0,LIST SUBQUERY 2
10,6,62,SEARCH stars USING INDEX person_index (person_id=?)
13,6,0,SCALAR SUBQUERY 1
17,13,56,SEARCH people USING COVERING INDEX name_index (name=?)
34,6,0,CREATE BLOOM FILTER


In [10]:
%sql DROP INDEX "person_index";

 * sqlite:///./movies.db?check_same_thread=false


[]

In [11]:
%%sql
CREATE INDEX "person_index"
ON "stars" ("person_id", "movie_id");

 * sqlite:///./movies.db?check_same_thread=false


[]

In [12]:
%%sql
EXPLAIN QUERY PLAN
SELECT "title" FROM "movies"
WHERE "id" IN (
    SELECT "movie_id" FROM "stars"
    WHERE "person_id" = (
        SELECT "id" FROM "people"
        WHERE "name" = 'Tom Hanks'
    )
);

 * sqlite:///./movies.db?check_same_thread=false


id,parent,notused,detail
2,0,91,SEARCH movies USING INTEGER PRIMARY KEY (rowid=?)
6,0,0,LIST SUBQUERY 2
9,6,56,SEARCH stars USING COVERING INDEX person_index (person_id=?)
12,6,0,SCALAR SUBQUERY 1
16,12,56,SEARCH people USING COVERING INDEX name_index (name=?)
32,6,0,CREATE BLOOM FILTER


In [13]:
%%sql
SELECT "title" FROM "movies"
WHERE "id" IN (
    SELECT "movie_id" FROM "stars"
    WHERE "person_id" = (
        SELECT "id" FROM "people"
        WHERE "name" = 'Tom Hanks'
    )
);

 * sqlite:///./movies.db?check_same_thread=false


title
Bachelor Party
Splash
The Man with One Red Shoe
Volunteers
Every Time We Say Goodbye
The Money Pit
Nothing in Common
Dragnet
Big
Punchline
